In [0]:
from pyspark.sql import functions as F

studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

interventions_df = spark.read.table(
    "clinical_trials.silver_interventions"
)

outcomes_df = spark.read.table(
    "clinical_trials.silver_outcomes"
)

locations_df = spark.read.table(
    "clinical_trials.silver_locations"
)

In [0]:
# Silver에서는 분석 및 정규화를 위해 Study-Condition,
# Study-Location 관계를 별도 row로 관리한다.
#
# Serving Layer에서는 application의 조회 편의를 위해
# 관련 condition/country를 study 단위의 array로 다시 집계한다.

condition_summary_df = (
    conditions_df
    .groupBy("nct_id")
    .agg(
        F.sort_array(
            # collect_set : collect_list()와 달리 중복을 제거하면서 배열로 모읍니다.
            F.collect_set("condition")
        ).alias("conditions")
    )
)

country_summary_df = (
    locations_df
    .filter(F.col("country").isNotNull())
    .groupBy("nct_id")
    .agg(
        F.sort_array(
            F.collect_set("country")
        ).alias("countries")
    )
)

intervention_summary_df = (
    interventions_df
    .filter(F.col("intervention_name").isNotNull())
    .groupBy("nct_id")
    .agg(
        F.sort_array(
            F.collect_set("intervention_name")
        ).alias("intervention_names")
    )
)

In [0]:
serving_study_search_df = (
    studies_df
    .join(
        condition_summary_df,
        on="nct_id",
        how="left"
    )
    .join(
        country_summary_df,
        on="nct_id",
        how="left"
    )
    .join(
        intervention_summary_df,
        on="nct_id",
        how="left"
    )
    .select(
        "nct_id",
        "brief_title",
        "official_title",
        "study_type",
        "phases",
        "overall_status",
        "enrollment_count",
        "start_date",
        "completion_date",
        "last_update_date",
        "has_results",
        "conditions",
        "countries",
        "intervention_names"
    )
)

display(
    serving_study_search_df.limit(20)
)

In [0]:
(
    serving_study_search_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "clinical_trials.serving_study_search"
    )
)

In [0]:
# QC
print(
    "Studies:",
    studies_df.count()
)

print(
    "Serving search:",
    serving_study_search_df.count()
)

print(
    "Duplicate NCT:",
    serving_study_search_df
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
# Serving study detail
# 특정 study 하나를 Agent나 CRA-RBM에서 열 때 사용할 데이터

# intervention, Oucome, Location을 한 study 단위로
intervention_detail_df = (
    interventions_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "intervention_order",
                "intervention_type",
                "intervention_name",
                "description",
                "arm_group_labels"
            )
        ).alias("interventions")
    )
)

outcome_detail_df = (
    outcomes_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "outcome_type",
                "outcome_order",
                "measure",
                "description",
                "time_frame"
            )
        ).alias("outcomes")
    )
)

location_detail_df = (
    locations_df
    .groupBy("nct_id")
    .agg(
        F.collect_list(
            F.struct(
                "location_order",
                "facility",
                "location_status",
                "city",
                "state",
                "country",
                "latitude",
                "longitude"
            )
        ).alias("locations")
    )
)

In [0]:
serving_study_detail_df = (
    studies_df
    .join(
        condition_summary_df,
        on="nct_id",
        how="left"
    )
    .join(
        intervention_detail_df,
        on="nct_id",
        how="left"
    )
    .join(
        outcome_detail_df,
        on="nct_id",
        how="left"
    )
    .join(
        location_detail_df,
        on="nct_id",
        how="left"
    )
    .select(
        "nct_id",
        "brief_title",
        "official_title",
        "study_type",
        "phases",
        "overall_status",

        "brief_summary",
        "who_masked",
        "eligibility_criteria",

        "enrollment_count",
        "enrollment_type",
        "allocation",
        "intervention_model",
        "primary_purpose",
        "masking",

        "start_date",
        "completion_date",
        "last_update_date",
        "has_results",
        "source",

        "conditions",
        "interventions",
        "outcomes",
        "locations",
    )
)

display(
    serving_study_detail_df.limit(5)
)

In [0]:
# 특정 NCT ID 조회 테스트

example_nct_id = (
    serving_study_detail_df
    .select("nct_id")
    .first()["nct_id"]
)

print(example_nct_id)

display(
    serving_study_detail_df
    .filter(
        F.col("nct_id") == example_nct_id
    )
)

In [0]:
(
    serving_study_detail_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "clinical_trials.serving_study_detail"
    )
)

print(
    "Serving Detail:",
    serving_study_detail_df.count()
)

print(
    "Duplicate:",
    serving_study_detail_df
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)